In [1]:
# -*- coding: utf-8 -*-
from ultralytics import YOLO
import os
from sklearn.model_selection import train_test_split

# 선택: data.yaml 자동 보정 위해 PyYAML 사용
import yaml

# ===== 경로 =====
DATASET_ROOT = '/Users/goodsee/dongwon/yolo_dataset'
IMG_DIR = os.path.join(DATASET_ROOT, 'train', 'images')  # 모든 이미지 여기에 그대로 둡니다
LBL_DIR = os.path.join(DATASET_ROOT, 'train', 'labels')  # 모든 라벨 여기에 그대로 둡니다
SPLIT_DIR = os.path.join(DATASET_ROOT, 'splits')         # 분할 리스트(.txt) 저장 폴더
DATA_YAML = os.path.join(DATASET_ROOT, 'data.yaml')

os.makedirs(SPLIT_DIR, exist_ok=True)

# ===== 이미지 목록 (라벨 존재하는 것만 채택 권장) =====
EX = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')
all_imgs = sorted(
    f for f in os.listdir(IMG_DIR)
    if f.lower().endswith(EX)
)

# 라벨 없는 샘플 제외(권장)
valid_imgs = []
for fn in all_imgs:
    base, _ = os.path.splitext(fn)
    if os.path.exists(os.path.join(LBL_DIR, base + '.txt')):
        valid_imgs.append(fn)

print(f'총 이미지 수: {len(all_imgs)} / 라벨 존재: {len(valid_imgs)}')

# ===== 분할(폴더 이동 없이 리스트만 생성) =====
# 재현 가능한 고정 분할을 위해 random_state 고정
train_files, val_files = train_test_split(valid_imgs, test_size=0.01, random_state=42)

train_list_path = os.path.join(SPLIT_DIR, 'train.txt')
val_list_path   = os.path.join(SPLIT_DIR, 'val.txt')

with open(train_list_path, 'w') as f:
    for fn in train_files:
        f.write(os.path.join(IMG_DIR, fn) + '\n')

with open(val_list_path, 'w') as f:
    for fn in val_files:
        f.write(os.path.join(IMG_DIR, fn) + '\n')

print(f'리스트 생성 완료: {train_list_path}, {val_list_path}')

# ===== data.yaml 갱신: train/val을 리스트 파일로 지정 =====
# 기존 data.yaml의 names, nc 등은 유지
data = {}
if os.path.exists(DATA_YAML):
    with open(DATA_YAML, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f) or {}

data['train'] = train_list_path
data['val']   = val_list_path
# data['path']는 없어도 됩니다. (절대경로 리스트를 쓰므로)
# data['names']가 없다면 classes 이름을 채워두세요. (권장)

with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

print('data.yaml 업데이트 완료:')
print('  train:', data['train'])
print('  val  :', data['val'])
if 'names' not in data:
    print("⚠️ 'names'가 비어 있습니다. data.yaml에 클래스명을 추가하면 더 좋아요.")

# ===== 라벨 파일 변환 (처음 한 번만 실행) =====
# 기존 sprocket 클래스들을 COCO 80개 클래스 뒤에 배치
# 0 -> 80 (sprocket), 1 -> 81 (sprocket_3), 2 -> 82 (sprocket_db30), 3 -> 83 (sprocket_z36)
LABEL_CONVERT_SCRIPT = os.path.join(DATASET_ROOT, 'convert_labels.py')
if os.path.exists(LABEL_CONVERT_SCRIPT):
    print("🔄 라벨 파일 변환 중...")
    import subprocess
    result = subprocess.run(['python3', LABEL_CONVERT_SCRIPT], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("⚠️ 경고:", result.stderr)
else:
    print("⚠️ convert_labels.py를 먼저 실행하세요!")

# ===== 모델 로드 & 학습 =====
# 기존 학습된 모델이 있으면 로드, 없으면 백본 모델 사용
TRAINED_MODEL_PATH = os.path.join(os.path.dirname(DATASET_ROOT), 'models', 'best.pt')
if os.path.exists(TRAINED_MODEL_PATH):
    print(f"📦 기존 학습된 모델 로드: {TRAINED_MODEL_PATH}")
    model = YOLO(TRAINED_MODEL_PATH)
    print('✅ 기존 모델에서 추가 학습 시작')
else:
    print("📦 백본 모델 사용 (처음 학습)")
    try:
        model = YOLO('yolo12n.pt')
        print('✅ YOLOv12 모델 사용')
    except Exception as e:
        print('YOLOv12 불가, YOLOv10 시도:', e)
        try:
            model = YOLO('yolov10n.pt')
            print('✅ YOLOv10 모델 사용')
        except Exception as e2:
            print('YOLOv10 불가, YOLOv8 사용:', e2)
            model = YOLO('yolov8n.pt')
            print('✅ YOLOv8 모델 사용')

# 일부 버전은 close_mosaic, cos_lr 미지원일 수 있어 예외 처리
# ⚠️ 중요: pretrained=True로 설정하여 백본의 사전 학습 가중치를 유지
#          이렇게 하면 다른 객체(person 등)와 sprocket을 구분할 수 있습니다.
train_kwargs = dict(
    data=DATA_YAML,
    epochs=50,
    patience=20,
    pretrained=True,   # ✅ 백본의 사전 학습 가중치 유지 (중요!)
    verbose=True,
    save=True,
    save_period=10,
    cache=True,
    workers=4,
    batch=16,
    imgsz=640,
    device='cpu',      # GPU 있으면 'cuda' 또는 0
    seed=42,
    lr0=0.001,         # 작은 LR로 미세튜닝 (백본이 망가지지 않도록)
    lrf=0.01,          # 최종 LR 비율
    close_mosaic=15,
    cos_lr=True
)

def train_with_fallback(model, kwargs):
    try:
        return model.train(**kwargs)
    except TypeError as e:
        msg = str(e)
        removed = []
        for key in ('close_mosaic', 'cos_lr'):
            if key in kwargs and key in msg:
                kwargs.pop(key, None)
                removed.append(key)
        if removed:
            print(f"ℹ️ 호환성 때문에 제거한 인자: {removed}")
            return model.train(**kwargs)
        raise

results = train_with_fallback(model, train_kwargs)
print("✅ 훈련 완료!")
try:
    print("📁 모델 저장 위치:", results.save_dir)
except Exception:
    pass


총 이미지 수: 1353 / 라벨 존재: 1353
리스트 생성 완료: /Users/goodsee/dongwon/yolo_dataset/splits/train.txt, /Users/goodsee/dongwon/yolo_dataset/splits/val.txt
data.yaml 업데이트 완료:
  train: /Users/goodsee/dongwon/yolo_dataset/splits/train.txt
  val  : /Users/goodsee/dongwon/yolo_dataset/splits/val.txt
🔄 라벨 파일 변환 중...
총 1353개의 라벨 파일 발견
✅ 0개의 라벨 파일이 변환되었습니다.
변환 매핑: {0: 80, 1: 81, 2: 82, 3: 83}
라벨 파일 변환 완료.

📦 기존 학습된 모델 로드: /Users/goodsee/dongwon/models/best.pt
✅ 기존 모델에서 추가 학습 시작
New https://pypi.org/project/ultralytics/8.3.228 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.218 🚀 Python-3.10.19 torch-2.2.0 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/goodsee/dongwon/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl

/opt/anaconda3/envs/dongwon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Plotting labels to /Users/goodsee/dongwon/yolo_dataset/runs/detect/train/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.001' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000114, momentum=0.9) with parameter groups 113 weight(decay=0.0), 120 weight(decay=0.0005), 119 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/goodsee/dongwon/yolo_dataset/runs/detect/train
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50         0G     0.6544      4.639      1.344         29        640: 100% ━━━━━━━━━━━━ 84/84 0.1it/s 11:304.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 0.8it/s 1.2s
                   all         14         14          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss 

In [ ]:
# -*- coding: utf-8 -*-
"""
YOLO Fine-tuning Script (전체 모델 학습)
- Stage 0: 리스트 분할 (폴더 이동 없음)
- 전체 모델 학습: 백본 포함, pretrained=True로 사전 학습 가중치 유지
  → 백본의 일반 객체 특징을 유지하면서 sprocket을 학습
  → 다른 객체와 sprocket을 구분할 수 있음
- 저장 경로: ./weight/full_train, 최종 ./weight/best.pt
"""

import os
import shutil
from sklearn.model_selection import train_test_split
import yaml

# 선택: GPU 자동 감지
try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

from ultralytics import YOLO

# =========================
# 사용자 설정
# =========================
DATASET_ROOT = '/Users/goodsee/dongwon/yolo_dataset'
IMG_DIR      = os.path.join(DATASET_ROOT, 'train', 'images')
LBL_DIR      = os.path.join(DATASET_ROOT, 'train', 'labels')
SPLIT_DIR    = os.path.join(DATASET_ROOT, 'splits')
DATA_YAML    = os.path.join(DATASET_ROOT, 'data.yaml')

VAL_RATIO    = 0.2
RANDOM_STATE = 42

# 로보플로우 등에서 오프라인 증강 이미지를 이미 포함했다면 True
USE_OFFLINE_AUG = True   # True면 온라인 증강 완전 OFF

# 결과 저장 루트
PROJECT_DIR = os.path.abspath('./weight')
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

# =========================
# 유틸
# =========================
EX = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp')

def auto_select_device():
    if TORCH_AVAILABLE and torch.cuda.is_available():
        return 'cuda:0'
    return 'cpu'

def list_labeled_images(img_dir, lbl_dir):
    all_imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(EX)])
    keep = []
    for fn in all_imgs:
        base, _ = os.path.splitext(fn)
        if os.path.exists(os.path.join(lbl_dir, base + '.txt')):
            keep.append(fn)
    return keep, all_imgs

def update_data_yaml(data_yaml_path, train_list_path, val_list_path):
    data = {}
    if os.path.exists(data_yaml_path):
        with open(data_yaml_path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f) or {}
    data['train'] = train_list_path
    data['val']   = val_list_path
    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)
    print('📝 data.yaml 업데이트:')
    print('  train:', data['train'])
    print('  val  :', data['val'])
    if 'names' not in data:
        print("⚠️ 'names'가 비어 있습니다. 클래스명이 필요하면 data.yaml에 추가하세요.")

def load_yolo_model():
    try:
        m = YOLO('yolo12n.pt')
        print('✅ YOLOv12 모델 사용')
        return m
    except Exception as e:
        print('YOLOv12 불가, YOLOv10 시도:', e)
        try:
            m = YOLO('yolov10n.pt')
            print('✅ YOLOv10 모델 사용')
            return m
        except Exception as e2:
            print('YOLOv10 불가, YOLOv8 사용:', e2)
            m = YOLO('yolov8n.pt')
            print('✅ YOLOv8 모델 사용')
            return m

def train_with_fallback(model, kwargs, manual_freeze_fn=None):
    """미지원 인자(TypeError) 발생 시 제거하고 재시도."""
    try:
        return model.train(**kwargs)
    except TypeError as e:
        msg = str(e)
        removed = []
        for key in ('close_mosaic', 'cos_lr', 'freeze'):
            if key in kwargs and key in msg:
                kwargs.pop(key, None)
                removed.append(key)
        if removed:
            print(f"ℹ️ 호환성 때문에 제거한 인자: {removed}")
            if 'freeze' in removed and callable(manual_freeze_fn):
                manual_freeze_fn(model)
            return model.train(**kwargs)
        raise

def manual_freeze_backbone_keep_detect(m):
    """freeze 인자 미지원 시: 백본 동결, Detect만 학습."""
    try:
        for p in m.model.parameters():
            p.requires_grad = False
        kept = 0
        for mod in m.model.modules():
            if mod.__class__.__name__.lower().endswith('detect'):
                for p in mod.parameters():
                    p.requires_grad = True
                kept += 1
        if kept == 0:
            for p in m.model.model[-1].parameters():
                p.requires_grad = True
            print("🔒 수동 동결: Detect 모듈 탐지 실패 → 마지막 블록만 학습으로 대체")
        else:
            print(f"🔒 수동 동결 완료: Detect 모듈 {kept}개 학습 유지")
    except Exception as ex:
        print("⚠️ 수동 동결 실패, 전체 파라미터 학습으로 진행:", ex)

def find_best_from_results(res):
    try:
        p = os.path.join(res.save_dir, 'weights', 'best.pt')
        if os.path.exists(p):
            return p
    except Exception:
        pass
    for name in ('stage1', 'stage2', 'exp'):
        cand = os.path.join(PROJECT_DIR, name, 'weights', 'best.pt')
        if os.path.exists(cand):
            return cand
    root = os.path.join('runs', 'detect')
    if os.path.isdir(root):
        subs = sorted([os.path.join(root, d) for d in os.listdir(root)], key=os.path.getmtime)
        if subs:
            cand = os.path.join(subs[-1], 'weights', 'best.pt')
            if os.path.exists(cand):
                return cand
    return None

# =========================
# Stage 0: 리스트 분할
# =========================
labeled_imgs, all_imgs = list_labeled_images(IMG_DIR, LBL_DIR)
print(f"총 이미지 수: {len(all_imgs)} / 라벨 존재: {len(labeled_imgs)}")

if len(labeled_imgs) >= 2 and VAL_RATIO > 0:
    train_files, val_files = train_test_split(
        labeled_imgs, test_size=VAL_RATIO, random_state=RANDOM_STATE
    )
else:
    train_files, val_files = labeled_imgs, []

train_list_path = os.path.join(SPLIT_DIR, 'train.txt')
val_list_path   = os.path.join(SPLIT_DIR, 'val.txt')
with open(train_list_path, 'w') as f:
    for fn in train_files:
        f.write(os.path.join(IMG_DIR, fn) + '\n')
with open(val_list_path, 'w') as f:
    for fn in val_files:
        f.write(os.path.join(IMG_DIR, fn) + '\n')

print(f"리스트 생성 완료: {train_list_path} ({len(train_files)}), {val_list_path} ({len(val_files)})")
update_data_yaml(DATA_YAML, train_list_path, val_list_path)

# =========================
# 증강 설정
# =========================
if USE_OFFLINE_AUG:
    augmentation_config = {
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
        'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'shear': 0.0, 'perspective': 0.0,
        'flipud': 0.0, 'fliplr': 0.0,
        'mosaic': 0.0, 'mixup': 0.0, 'copy_paste': 0.0,
    }
else:
    augmentation_config = {
        'hsv_h': 0.015, 'hsv_s': 0.4, 'hsv_v': 0.25,
        'degrees': 5.0, 'translate': 0.05, 'scale': 0.15, 'shear': 1.0, 'perspective': 0.0,
        'flipud': 0.0, 'fliplr': 0.5,
        'mosaic': 0.2, 'mixup': 0.0, 'copy_paste': 0.0,
    }

print("📊 데이터 증강 설정:")
for k, v in augmentation_config.items():
    print(f"  - {k}: {v}")

DEVICE = auto_select_device()
print(f"🖥️ 사용 디바이스: {DEVICE}")

# =========================
# 공통 학습 파라미터 (pretrained 제외!)
# =========================
COMMON_TRAIN_KW = dict(
    data=DATA_YAML,
    patience=20,
    verbose=True,
    save=True,
    save_period=10,
    cache=True,
    workers=4,
    batch=16,
    imgsz=640,
    device=DEVICE,
    seed=RANDOM_STATE,
    cos_lr=True,   # 미지원이면 fallback에서 제거됨
    **augmentation_config
)
# mosaic이 켜진 경우에만 close_mosaic 사용
if augmentation_config.get('mosaic', 0.0) > 0.0:
    COMMON_TRAIN_KW['close_mosaic'] = 15

# =========================
# 모델 로드
# =========================
model = load_yolo_model()

# =========================
# 전체 모델 학습 (백본 포함, pretrained=True로 사전 학습 가중치 유지)
# =========================
print("\n=== 전체 모델 학습 (백본 포함, 사전 학습 가중치 유지) ===")
print("⚠️ Stage 1(백본 동결)을 건너뛰고 전체 모델을 학습합니다.")
print("   이렇게 하면 백본의 일반 객체 특징을 유지하면서 sprocket을 학습할 수 있습니다.")

train_kwargs_full = dict(
    epochs=50,            # 전체 학습 epoch
    pretrained=True,      # ✅ 백본의 사전 학습 가중치 유지 (중요!)
    lr0=0.001,            # 작은 LR로 미세튜닝 (백본이 망가지지 않도록)
    lrf=0.01,             # 최종 LR 비율
    project=PROJECT_DIR,
    name='full_train',
    exist_ok=True,
    **COMMON_TRAIN_KW
)
results = train_with_fallback(model, train_kwargs_full)

# 최종 best 복사
final_best = os.path.join(PROJECT_DIR, 'full_train', 'weights', 'best.pt')
if not os.path.exists(final_best):
    final_best = find_best_from_results(results)

if final_best and os.path.exists(final_best):
    dst = os.path.join(PROJECT_DIR, 'best.pt')
    shutil.copy2(final_best, dst)
    print(f"✅ 최종 가중치 복사 완료: {dst}")
else:
    print("⚠️ 최종 best.pt를 찾지 못했습니다. full_train 폴더를 확인하세요.")

print("\n✅ 전체 모델 학습 완료!")
print(f"📁 학습 결과: {os.path.join(PROJECT_DIR, 'full_train')}")
print(f"📁 최종 best: {os.path.join(PROJECT_DIR, 'best.pt')}")
print("\n💡 백본의 사전 학습 가중치를 유지하면서 학습했으므로,")
print("   다른 객체와 sprocket을 구분할 수 있어야 합니다.")

print("\n[참고] 예측 시 중복 박스가 보이면 conf를 0.45~0.6으로 올리고, iou를 0.5로 낮춰보세요.")
print("예: model2.predict(source='path/to/img_or_dir', conf=0.5, iou=0.5)")


총 이미지 수: 312 / 라벨 존재: 312
리스트 생성 완료: /Users/goodsee/dongwon/yolo_dataset/splits/train.txt (249), /Users/goodsee/dongwon/yolo_dataset/splits/val.txt (63)
📝 data.yaml 업데이트:
  train: /Users/goodsee/dongwon/yolo_dataset/splits/train.txt
  val  : /Users/goodsee/dongwon/yolo_dataset/splits/val.txt
📊 데이터 증강 설정:
  - hsv_h: 0.0
  - hsv_s: 0.0
  - hsv_v: 0.0
  - degrees: 0.0
  - translate: 0.0
  - scale: 0.0
  - shear: 0.0
  - perspective: 0.0
  - flipud: 0.0
  - fliplr: 0.0
  - mosaic: 0.0
  - mixup: 0.0
  - copy_paste: 0.0
🖥️ 사용 디바이스: cpu
✅ YOLOv12 모델 사용

=== Stage 1: Detect Head만 학습 (백본 동결) ===
New https://pypi.org/project/ultralytics/8.3.225 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.218 🚀 Python-3.10.19 torch-2.2.0 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=

/opt/anaconda3/envs/dongwon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Plotting labels to /Users/goodsee/dongwon/yolo_dataset/weight/stage1/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 113 weight(decay=0.0), 120 weight(decay=0.0005), 119 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/goodsee/dongwon/yolo_dataset/weight/stage1
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/20         0G     0.9003      2.805      1.735          9        640: 100% ━━━━━━━━━━━━ 16/16 0.1it/s 1:584.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 0.2it/s 12.8s1.7s
                   all         63         63    0.00333          1      0.966      0.552

      Epoch    GPU_mem   box_loss   cls_loss   dfl_lo

TypeError: dict() got multiple values for keyword argument 'pretrained'